In [4]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\volin\OneDrive\Desktop\CodingChallenge\data\data")

files = list(DATA_DIR.glob("*.csv"))

print(f"Total CSV files: {len(files)}")

accounts = [f for f in files if f.name.lower() == "accounts.csv"]
daily = [f for f in files if f.name.lower().startswith("daily_")]
monthly = [f for f in files if f.name.lower().startswith("monthly_")]

print(f"Accounts files: {len(accounts)}")
print(f"Daily files: {len(daily)}")
print(f"Monthly files: {len(monthly)}")

print("\nDaily examples:")
for file in sorted(daily)[:5]:
    print(file.name)

print("\nMonthly examples:")
for file in sorted(monthly)[:5]:
    print(file.name)

Total CSV files: 376
Accounts files: 1
Daily files: 362
Monthly files: 13

Daily examples:
daily_20241126.csv
daily_20241127.csv
daily_20241128.csv
daily_20241129.csv
daily_20241130.csv

Monthly examples:
monthly_202411.csv
monthly_202412.csv
monthly_202501.csv
monthly_202502.csv
monthly_202503.csv


In [7]:


import polars as pl

accounts_file = accounts[0]
daily_file = sorted(daily)[0]
monthly_file = sorted(monthly)[0]

for label, file in [
    ("Accounts", accounts_file),
    ("Daily Status", daily_file),
    ("Monthly Status", monthly_file)
]:
    print(f"\n--- {label}: {file.name} ---")

    df = pl.read_csv(file)

    print("Columns:", df.columns)
    print("Rows:", df.height)
    print("Schema:")
    print(df.schema)
    print("\nFirst 5 rows:")
    print(df.head())


--- Accounts: accounts.csv ---
Columns: ['account_id', 'name', 'address']
Rows: 501
Schema:
Schema({'account_id': Int64, 'name': String, 'address': String})

First 5 rows:
shape: (5, 3)
┌────────────┬────────────────┬──────────────────────────────┐
│ account_id ┆ name           ┆ address                      │
│ ---        ┆ ---            ┆ ---                          │
│ i64        ┆ str            ┆ str                          │
╞════════════╪════════════════╪══════════════════════════════╡
│ 95948      ┆ James Davis    ┆ 45 Riverside Drive, Cork     │
│ 83620      ┆ David Wilson   ┆ 122 Pine Road, Waterford     │
│ 43314      ┆ Michael Miller ┆ 13 Elm Street, Belfast       │
│ 42247      ┆ Alex Jones     ┆ 21 Cedar Lane, Waterford     │
│ 17346      ┆ Daniel Wilson  ┆ 37 Riverside Drive, Limerick │
└────────────┴────────────────┴──────────────────────────────┘

--- Daily Status: daily_20241126.csv ---
Columns: ['account', 'queue', 'status', 'changed_datetime']
Rows: 7
Schema:
Sc

In [8]:
def inspect_files(files, expected_columns):
    issues = []

    for file in sorted(files):
        df = pl.read_csv(file)

        if df.columns != expected_columns:
            issues.append({
                "file": file.name,
                "issue": "Unexpected columns",
                "columns": df.columns
            })

    return issues


daily_issues = inspect_files(
    daily,
    ["account", "queue", "status", "changed_datetime"]
)

monthly_issues = inspect_files(
    monthly,
    ["account", "queue", "status", "month"]
)

print("Daily schema issues:")
print(daily_issues)

print("\nMonthly schema issues:")
print(monthly_issues)

Daily schema issues:
[]

Monthly schema issues:
[]


In [9]:
import re
from datetime import datetime

def validate_filenames(files, pattern):
    issues = []

    for file in files:
        match = re.match(pattern, file.stem)

        if not match:
            issues.append((file.name, "Invalid filename"))
            continue

        date_value = match.group(1)

        try:
            if pattern == r"daily_(\d{8})":
                datetime.strptime(date_value, "%Y%m%d")
            else:
                datetime.strptime(date_value, "%Y%m")
        except ValueError:
            issues.append((file.name, "Invalid date in filename"))

    return issues


print("Daily filename issues:")
print(validate_filenames(daily, r"daily_(\d{8})"))

print("\nMonthly filename issues:")
print(validate_filenames(monthly, r"monthly_(\d{6})"))

Daily filename issues:
[]

Monthly filename issues:
[]


In [11]:
def check_nulls(files):
    results = []

    for file in sorted(files):
        df = pl.read_csv(file)
        nulls = df.null_count()

        for column in df.columns:
            count = nulls[column][0]

            if count > 0:
                results.append({
                    "file": file.name,
                    "column": column,
                    "null_count": count
                })

    return results


daily_nulls = check_nulls(daily)
monthly_nulls = check_nulls(monthly)

print("Daily nulls:")
print(daily_nulls)

print("\nMonthly nulls:")
print(monthly_nulls)

accounts_df = pl.read_csv(accounts_file)

print("Accounts nulls:")
print(accounts_df.null_count())

Daily nulls:
[]

Monthly nulls:
[]
Accounts nulls:
shape: (1, 3)
┌────────────┬──────┬─────────┐
│ account_id ┆ name ┆ address │
│ ---        ┆ ---  ┆ ---     │
│ u32        ┆ u32  ┆ u32     │
╞════════════╪══════╪═════════╡
│ 0          ┆ 0    ┆ 0       │
└────────────┴──────┴─────────┘


In [12]:
account_ids = set(accounts_df["account_id"].to_list())


def find_unknown_accounts(files):
    unknown = {}

    for file in sorted(files):
        df = pl.read_csv(file)

        missing = (
            set(df["account"].to_list())
            - account_ids
        )

        if missing:
            unknown[file.name] = missing

    return unknown


daily_unknown = find_unknown_accounts(daily)
monthly_unknown = find_unknown_accounts(monthly)

print("Unknown accounts in Daily files:")
print(daily_unknown)

print("\nUnknown accounts in Monthly files:")
print(monthly_unknown)


Unknown accounts in Daily files:
{}

Unknown accounts in Monthly files:
{}


In [13]:
def find_duplicates(files, columns):
    duplicates = []

    for file in sorted(files):
        df = pl.read_csv(file)

        dup = (
            df.group_by(columns)
            .agg(pl.len().alias("count"))
            .filter(pl.col("count") > 1)
        )

        if dup.height > 0:
            duplicates.append({
                "file": file.name,
                "duplicate_groups": dup.height,
                "duplicate_rows": dup["count"].sum()
            })

    return duplicates


daily_duplicates = find_duplicates(
    daily,
    ["account", "queue", "status", "changed_datetime"]
)

monthly_duplicates = find_duplicates(
    monthly,
    ["account", "queue", "status", "month"]
)

print("Daily duplicates:")
print(daily_duplicates)

print("\nMonthly duplicates:")
print(monthly_duplicates)

Daily duplicates:
[]

Monthly duplicates:
[]


In [15]:
def find_same_timestamp_changes(files):
    results = []

    for file in sorted(files):
        df = pl.read_csv(file)

        dup = (
            df.group_by(["account", "changed_datetime"])
            .agg(pl.len().alias("count"))
            .filter(pl.col("count") > 1)
        )

        if dup.height > 0:
            results.append({
                "file": file.name,
                "groups": dup.height,
                "rows": dup["count"].sum()
            })

    return results


same_timestamp = find_same_timestamp_changes(daily)

print(same_timestamp)

[]


In [21]:
daily_df = pl.concat(
    [
        pl.read_csv(
            file,
            schema_overrides={
                "account": pl.Int64,
                "changed_datetime": pl.String
            }
        )
        for file in sorted(daily)
    ],
    how="vertical"
)

print(daily_df.shape)
print(daily_df.schema)
print(daily_df.head())

(3625, 4)
Schema({'account': Int64, 'queue': String, 'status': String, 'changed_datetime': String})
shape: (5, 4)
┌─────────┬─────────────┬─────────────┬─────────────────────┐
│ account ┆ queue       ┆ status      ┆ changed_datetime    │
│ ---     ┆ ---         ┆ ---         ┆ ---                 │
│ i64     ┆ str         ┆ str         ┆ str                 │
╞═════════╪═════════════╪═════════════╪═════════════════════╡
│ 26686   ┆ PAYING      ┆ PAY PROP    ┆ 2024-11-26 06:57:35 │
│ 48388   ┆ OTHER       ┆ ARCHIVED    ┆ 2024-11-26 09:28:43 │
│ 81634   ┆ COLLECTIONS ┆ CANCEL2     ┆ 2024-11-26 12:58:14 │
│ 39776   ┆ INSOLVENCY  ┆ INS4        ┆ 2024-11-26 17:07:59 │
│ 70437   ┆ LEGAL       ┆ LEG VERIF 2 ┆ 2024-11-26 19:00:23 │
└─────────┴─────────────┴─────────────┴─────────────────────┘


In [22]:
from datetime import datetime

def read_monthly(file):
    df = pl.read_csv(
        file,
        schema_overrides={"account": pl.Int64, "month": pl.Int64}
    )

    month_date = datetime.strptime(
        file.stem.split("_")[1],
        "%Y%m"
    ).date()

    return df.with_columns(
        pl.lit(month_date).cast(pl.Date).alias("month")
    )

monthly_df = pl.concat(
    [read_monthly(file) for file in sorted(monthly)],
    how="vertical"
)

print(monthly_df.shape)
print(monthly_df.schema)
print(monthly_df.head())

(5476, 4)
Schema({'account': Int64, 'queue': String, 'status': String, 'month': Date})
shape: (5, 4)
┌─────────┬─────────────┬─────────────┬────────────┐
│ account ┆ queue       ┆ status      ┆ month      │
│ ---     ┆ ---         ┆ ---         ┆ ---        │
│ i64     ┆ str         ┆ str         ┆ date       │
╞═════════╪═════════════╪═════════════╪════════════╡
│ 26686   ┆ PAYING      ┆ PAY PROP    ┆ 2024-11-01 │
│ 48388   ┆ OTHER       ┆ ARCHIVED    ┆ 2024-11-01 │
│ 81634   ┆ COLLECTIONS ┆ CANCEL2     ┆ 2024-11-01 │
│ 39776   ┆ INSOLVENCY  ┆ INS4        ┆ 2024-11-01 │
│ 70437   ┆ LEGAL       ┆ LEG VERIF 2 ┆ 2024-11-01 │
└─────────┴─────────────┴─────────────┴────────────┘


In [23]:
print(
    daily_df.join(
        accounts_df.select("account_id"),
        left_on="account",
        right_on="account_id",
        how="anti"
    ).height
)

print(
    monthly_df.join(
        accounts_df.select("account_id"),
        left_on="account",
        right_on="account_id",
        how="anti"
    ).height
)

0
0


In [24]:
daily_df = daily_df.rename({"account": "account_id"})
monthly_df = monthly_df.rename({"account": "account_id"})

In [27]:
daily_df.filter(
    ~pl.col("changed_datetime").str.contains(r"^\d{4}-\d{2}-\d{2}")
).select("changed_datetime")

print(
    daily_df.select(
        pl.when(pl.col("changed_datetime").str.contains(r"^\d{4}-\d{2}-\d{2}"))
        .then(pl.lit("standard"))
        .when(pl.col("changed_datetime").str.contains(r"^\d{2}-\d{2}-\d{2}"))
        .then(pl.lit("short"))
        .when(pl.col("changed_datetime").str.contains(r"^\d+\.\d+$"))
        .then(pl.lit("excel_serial"))
        .otherwise(pl.lit("unknown"))
        .alias("format")
    )
    .group_by("format")
    .len()
)

shape: (3, 2)
┌──────────────┬──────┐
│ format       ┆ len  │
│ ---          ┆ ---  │
│ str          ┆ u32  │
╞══════════════╪══════╡
│ short        ┆ 13   │
│ excel_serial ┆ 14   │
│ standard     ┆ 3598 │
└──────────────┴──────┘


In [29]:
daily_df = daily_df.with_columns(
    pl.when(pl.col("changed_datetime").str.contains(r"^\d{4}-\d{2}-\d{2}"))
    .then(
        pl.col("changed_datetime").str.to_datetime("%Y-%m-%d %H:%M:%S")
    )
    .when(pl.col("changed_datetime").str.contains(r"^\d{2}-\d{2}-\d{2}"))
    .then(
        pl.col("changed_datetime").str.to_datetime("%d-%m-%y %H:%M")
    )
    .when(pl.col("changed_datetime").str.contains(r"^\d+\.\d+$"))
    .then(
        pl.lit("1899-12-30").str.to_datetime("%Y-%m-%d")
        + (
            pl.col("changed_datetime").cast(pl.Float64) * 86_400_000_000
        ).cast(pl.Duration("us"))
    )
    .alias("changed_datetime")
)

In [30]:
print(daily_df.schema)
print(daily_df.select("changed_datetime").null_count())
print(daily_df.select("changed_datetime").head())

Schema({'account_id': Int64, 'queue': String, 'status': String, 'changed_datetime': Datetime(time_unit='us', time_zone=None)})
shape: (1, 1)
┌──────────────────┐
│ changed_datetime │
│ ---              │
│ u32              │
╞══════════════════╡
│ 0                │
└──────────────────┘
shape: (5, 1)
┌─────────────────────┐
│ changed_datetime    │
│ ---                 │
│ datetime[μs]        │
╞═════════════════════╡
│ 2024-11-26 06:57:35 │
│ 2024-11-26 09:28:43 │
│ 2024-11-26 12:58:14 │
│ 2024-11-26 17:07:59 │
│ 2024-11-26 19:00:23 │
└─────────────────────┘


In [31]:
print(
    daily_df.select(
        pl.col("changed_datetime").min().alias("min_datetime"),
        pl.col("changed_datetime").max().alias("max_datetime")
    )
)

shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_datetime        ┆ max_datetime        │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2024-11-26 06:57:35 ┆ 2025-11-26 23:42:04 │
└─────────────────────┴─────────────────────┘


In [32]:
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

In [33]:
accounts_df.write_csv(OUTPUT_DIR / "accounts_processed.csv")
daily_df.write_csv(OUTPUT_DIR / "daily_status_processed.csv")
monthly_df.write_csv(OUTPUT_DIR / "monthly_status_processed.csv")

In [34]:
print(list(OUTPUT_DIR.glob("*.csv")))

[WindowsPath('C:/Users/volin/OneDrive/Desktop/CodingChallenge/data/processed/accounts_processed.csv'), WindowsPath('C:/Users/volin/OneDrive/Desktop/CodingChallenge/data/processed/daily_status_processed.csv'), WindowsPath('C:/Users/volin/OneDrive/Desktop/CodingChallenge/data/processed/monthly_status_processed.csv')]


In [35]:
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

print(OUTPUT_DIR)

C:\Users\volin\OneDrive\Desktop\CodingChallenge\data\processed


In [37]:
accounts_df.write_csv(OUTPUT_DIR / "accounts_processed.csv")
print((OUTPUT_DIR / "accounts_processed.csv").exists())

True


In [38]:
daily_df.write_csv(OUTPUT_DIR / "daily_status_processed.csv")
print((OUTPUT_DIR / "daily_status_processed.csv").exists())

True


In [39]:
monthly_df.write_csv(OUTPUT_DIR / "monthly_status_processed.csv")
print((OUTPUT_DIR / "monthly_status_processed.csv").exists())

True


In [40]:
for file in OUTPUT_DIR.glob("*.csv"):
    df = pl.read_csv(file)
    print(file.name, df.shape)

accounts_processed.csv (501, 3)
daily_status_processed.csv (3625, 4)
monthly_status_processed.csv (5476, 4)
